# PairPulse reproduction notebook

This notebook reproduces the single-pulse validation and separated double-pulse spectrum in the manuscript. Run it from the PairPulse project root after installing the `plots` dependencies, or adjust `PROJECT` below. All calculations use natural units with `m=q=1`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path.cwd()
if not (PROJECT / 'src' / 'pairpulse').exists():
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / 'src'))
from pairpulse import (PulseTrain, SauterPulse, integrated_density,
                       sauter_exact_occupation, solve_dirac_mode,
                       solve_qke_mode)
print('Project root:', PROJECT)

## Exact single-pulse benchmark

The pulse is `E0=0.3`, `tau=2`. The exact Sauter result is compared with direct Dirac-mode evolution and the quantum-kinetic equations.

In [ ]:
single = SauterPulse(amplitude=0.3, duration=2.0)
p0 = 0.0
exact0 = sauter_exact_occupation(p0, single)
dirac0 = solve_dirac_mode(p0, single)
qke0 = solve_qke_mode(p0, single)
print(f'exact      = {exact0:.15g}')
print(f'Dirac      = {dirac0.occupation:.15g}; norm error = {dirac0.diagnostic:.3e}')
print(f'QKE        = {qke0.occupation:.15g}; invariant error = {qke0.diagnostic:.3e}')
print(f'|Dirac-exact| = {abs(dirac0.occupation-exact0):.3e}')
print(f'|QKE-exact|   = {abs(qke0.occupation-exact0):.3e}')

In [ ]:
p = np.linspace(-1.5, 1.5, 41)
f_dirac = np.array([solve_dirac_mode(float(x), single).occupation for x in p])
f_qke = np.array([solve_qke_mode(float(x), single).occupation for x in p])
f_exact = np.array([sauter_exact_occupation(float(x), single) for x in p])
plt.figure(figsize=(6.3, 4.1))
plt.semilogy(p, f_exact, color='black', lw=2, label='exact Sauter result')
plt.semilogy(p, f_dirac, 'o', ms=3.5, mfc='none', label='Dirac modes')
plt.semilogy(p, f_qke, 'x', ms=4.2, label='quantum-kinetic modes')
plt.xlabel(r'canonical momentum $p/m$'); plt.ylabel(r'occupation $f_p$')
plt.legend(frameon=False); plt.tight_layout(); plt.show()
print('maximum absolute errors:', np.max(np.abs(f_dirac-f_exact)), np.max(np.abs(f_qke-f_exact)))

## Separated pulse train

The following calculation has no analytic benchmark in the package. Agreement between the two formulations is the numerical cross-check; it does not imply a new physical mechanism.

In [ ]:
double = PulseTrain((SauterPulse(0.30, 1.1, -3.5),
                     SauterPulse(0.30, 1.1,  3.5)))
pd = np.linspace(-1.8, 1.8, 41)
fd = np.array([solve_dirac_mode(float(x), double).occupation for x in pd])
fk = np.array([solve_qke_mode(float(x), double).occupation for x in pd])
plt.figure(figsize=(6.3, 4.1))
plt.plot(pd, fd, 'o-', ms=3.2, lw=1.1, label='Dirac modes')
plt.plot(pd, fk, 'x--', ms=3.6, lw=1.0, label='quantum-kinetic modes')
plt.xlabel(r'canonical momentum $p/m$'); plt.ylabel(r'occupation $f_p$')
plt.legend(frameon=False); plt.tight_layout(); plt.show()
print('finite-window densities:', integrated_density(pd, fd), integrated_density(pd, fk))

To regenerate all figures, CSV files, convergence data, and run metadata exactly as used in the manuscript, run `python scripts/reproduce.py --output results --points 41` from the project root.